In [25]:
import pandas as pd
from pathlib import Path

#Crear estructura de carpetas
BASE = Path('content/EP2_guia06')
carpetas = [
    'datos/geo',
    'documentos/geo',
    'evidencias/geo',
    'visualizaciones',
    'notebooks'
]

for c in carpetas:
  (BASE / c).mkdir(parents=True, exist_ok=True)

(BASE / 'README.md').touch()

print("Estructura creada")
for i in sorted(BASE.rglob('*')):
  nivel = len(i.relative_to(BASE).parts) - 1
  print(' ' * nivel + i.name + ('/' if i.is_dir() else ''))

Estructura creada
README.md
datos/
 geo/
  .ipynb_checkpoints/
  centros_geograficos.csv
  operacion_centros.csv
documentos/
 geo/
evidencias/
 geo/
notebooks/
visualizaciones/


In [26]:
#Crear estructuras de carpetas
ruta_operacion = BASE / "datos" / "geo" / "operacion_centros.csv"
ruta_centros = BASE / "datos" / "geo" / "centros_geograficos.csv"

#Leer los archivos
df_operacion = pd.read_csv(ruta_operacion)
df_centros = pd.read_csv(ruta_centros)

#leer csvs
print(df_operacion.head())
print(df_centros.head())

        fecha centro_id     region  capacidad_paquetes  demanda_paquetes  \
0  2026-01-05       C01  Occidente                1092              1014   
1  2026-01-05       C02  Occidente                 886               891   
2  2026-01-05       C03  Occidente                 952               739   
3  2026-01-05       C04     Centro                1301              1319   
4  2026-01-05       C05     Centro                1004              1197   

   campana_programada  
0                   0  
1                   0  
2                   0  
3                   0  
4                   0  
  centro_id  nombre_centro  municipio     region  latitud  longitud
0       C01  Hub Santa Ana  Santa Ana  Occidente  13.9942  -89.5597
1       C02    Hub Metapán    Metapán  Occidente  14.3333  -89.4500
2       C03   Hub Acajutla   Acajutla  Occidente  13.5917  -89.8281
3       C04  Hub Soyapango  Soyapango     Centro -89.1370   13.7102
4       C05      Hub Apopa      Apopa     Centro  13.8072  

In [27]:
#calcular el deficit por cada fila
df_operacion['deficit_paquetes'] = (df_operacion['demanda_paquetes'] - df_operacion['capacidad_paquetes']).clip(lower=0)

#calcular el deficit acumulado por centro
df_deficit = df_operacion.groupby('centro_id')['deficit_paquetes'].sum().reset_index()
df_deficit.rename(columns={'deficit_paquetes': 'deficit_total_paquetes'}, inplace=True)

#asociar con los centros geograficos
df_merged = pd.merge(df_centros, df_deficit, on='centro_id', how='left')

In [28]:
#convertir coordenadas a numericas
df_merged['latitud'] = pd.to_numeric(df_merged['latitud'], errors='coerce')
df_merged['longitud'] = pd.to_numeric(df_merged['longitud'], errors='coerce')

#verificar plausibilidad territorial
mask_valida = (
    df_merged['latitud'].between(13, 15) &
    df_merged['longitud'].between(-91, -87)
)

#separar validos e invalidos
df_validos = df_merged[mask_valida]
df_excluidos = df_merged[~mask_valida]

In [29]:
#Exportar csv final con las columnas requeridas
df_final = df_validos[['centro_id', 'latitud', 'longitud', 'deficit_total_paquetes']]
ruta_salida_geo = BASE / "datos" / "geo" / "geo_validao.csv"
df_final.to_csv(ruta_salida_geo, index=False, encoding='utf-8')

print(f"Archivo listo para ArcGIS: {ruta_salida_geo}")

Archivo listo para ArcGIS: content/EP2_guia06/datos/geo/datasetGeo_Final.csv


In [30]:
#depuracion
print("--- REPORTE ---")
print(f"Total de centros originales: {len(df_centros)}")
print(f"Centros despues de la validacion (válidos): {len(df_validos)}")
print(f"Centros excluidos por coordenadas malas: {len(df_excluidos)}")
print(f"Centros sin datos de déficit (falló el merge): {df_validos['deficit_total_paquetes'].isna().sum()}")
print("---------------")

--- REPORTE ---
Total de centros originales: 9
Centros despues de la validacion (válidos): 8
Centros excluidos por coordenadas malas: 1
Centros sin datos de déficit (falló el merge): 0
---------------


# **Justificacion tecnica**
Se validaron 8 centros operativos dentro del rango territorial de El Salvador y se excluyo 1 por coordenadas incorrectas. El mayor deficit validado se presenta en el centro C07 con 36,725 paquetes. El riesgo de procesar y mostrar coordenadas erroneas es proyectar ubicaciones fuera del pais lo cual invalida analisar de forma correcta los datos.
link mapa: https://arcg.is/05fzCu4